In [3]:
import pandas as pd
bikes_raw = pd.read_csv('../data/bikes_clean.csv')
bikes = bikes_raw.copy()

In [64]:
print(bikes.shape)

(400, 16)


# 1 a)

In [84]:
print(bikes.price.notna().sum())
print(bikes.groupby('station').agg(
    num_bikes=('price', 'count'),
    median_price=('price', 'median')
).sort_values('median_price', ascending=False).round(2))

350
             num_bikes  median_price
station                             
Kungsholmen         73         742.0
Södermalm           87         720.0
Östermalm           48         709.5
Hammarby            47         609.0
Central             95         596.0


# b)

In [66]:
ebikes = bikes.loc[bikes.bike_type == 'E-bike']

In [85]:
print(len(ebikes))
print(ebikes.price.notna().sum())
print(ebikes.groupby('station').agg(
    num_ebikes=('price', 'count'),
    median_price=('price', 'median')
).sort_values('median_price', ascending=False).round(2))

124
108
             num_ebikes  median_price
station                              
Kungsholmen          22        1554.0
Södermalm            25        1220.0
Central              27        1143.5
Östermalm            19        1087.5
Hammarby             15         711.5


# c)

In [68]:
bikes.groupby('station').agg(
    mean_ebikes=('bike_type', lambda x: ((x == 'E-bike').mean()*100).round(1))
).sort_values('mean_ebikes', ascending=False)

,mean_ebikes
station,
Östermalm,38.6
Hammarby,31.5
Södermalm,30.5
Kungsholmen,29.5
Central,28.3


What we know for sure: 1 + 1 = 2. What we don't know: We still can't tell the answer to the universe and everything.
As for our tables above, they for sure give us some information and for sure does not give us all information. For example, the median price in Hammarby and Central are slightly lower than the other areas, but could be other factors playing a role...

# 2)

In [81]:
bikes.loc[bikes.rental_days < 0]

,rental_id,station,bike_type,helmets_text,is_member,insured,paid_online,status,start_date,end_date,price,rating,rental_days,has_rating,child_helmet,helmets_count
6,R-5077,Central,City bike,2 helmets,False,True,False,Completed,NaN,2026-07-18,476.0,5.0,-1.0,True,False,2
196,R-5217,Kungsholmen,E-bike,No helmets,True,False,True,Completed,NaN,2026-06-18,334.0,4.0,-2.0,True,False,0


In [82]:
bikes.loc[bikes.rental_days < 0, 'start_date'] = None

In [83]:
print(bikes.start_date.isna().sum())

4


2 dates had invalid start_dates since before and another 2 start_dates were after then end_date

Is there another table or log that could give us the correct start_dates and end_dates?

# 3)

In [6]:
def clean_bikes():
    bikes_raw = pd.read_csv('../data/bike_rentals.csv')
    bikes = bikes_raw.copy()
    bikes = bikes.dropna(axis=1, how='all')
    sprice = bikes.price.str.replace('SEK', '').str.replace(' ', '').str.replace(',', '')
    bikes.price = sprice.astype(float)
    tf_columns = [col for col in bikes.columns if set(bikes[col].dropna().unique()) <= {'t', 'f'}]
    for col in tf_columns:
        bikes[col] = bikes[col].map({'t': True, 'f': False})
    for col in ['start_date', 'end_date']:
        bikes[col] = pd.to_datetime(bikes[col], errors='coerce')
    bikes['rental_days'] = (bikes.end_date - bikes.start_date).dt.days
    bikes['has_rating'] = bikes.rating.isna() == False
    bikes['child_helmet'] = bikes.helmets_text.str.lower().str.contains('child')
    bikes['helmets_count'] = bikes.helmets_text.str.lower().str.extract(r'([\d]+)', expand=False).fillna(0).astype(int)
    bikes.station = bikes.station.str.strip().str.title()
    bikes = bikes.drop_duplicates().reset_index(drop=True)
    bikes.to_csv('../data/bike_rentals_clean.csv', index=False)
    

In [ ]:
clean_bikes()

In [14]:
bikes_clean = pd.read_csv('../data/bike_rentals_clean.csv', parse_dates=['start_date', 'end_date'])

In [15]:
bikes = bikes_clean.copy()

In [16]:
bikes.shape

(400, 16)

In [19]:
bikes.info()

<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   rental_id      400 non-null    str           
 1   station        400 non-null    str           
 2   bike_type      400 non-null    str           
 3   helmets_text   396 non-null    str           
 4   is_member      400 non-null    bool          
 5   insured        397 non-null    object        
 6   paid_online    400 non-null    bool          
 7   status         400 non-null    str           
 8   start_date     398 non-null    datetime64[us]
 9   end_date       400 non-null    datetime64[us]
 10  price          350 non-null    float64       
 11  rating         286 non-null    float64       
 12  rental_days    398 non-null    float64       
 13  has_rating     400 non-null    bool          
 14  child_helmet   400 non-null    bool          
 15  helmets_count  400 non-null    int

In [12]:
bikes.insured.head(3)

0     True
1     True
2    False
Name: insured, dtype: object

insured is object cause not all values are true/false

In [21]:
bikes.insured.value_counts(dropna=False)

insured
True     217
False    180
NaN        3
Name: count, dtype: int64

# 4)

* empty columns: removed
* price chaged from string of format 'SEK 1,200.50' to float like 1200.5
* columns with values 't' and 'f' (and NaN) changed to values True and False (and NaN are still NaN)
* Added columns 'rental_days', 'has_rating', 'child_helmet', 'helmets_count' derived from existing columns
* normalized values in station column
* dropped duplicated rows

# 5)

What's the average rental days per bike type and how many was counted for each type?

In [25]:
bikes.groupby('bike_type').agg(
    num= ('rental_days', 'count'),
    avg_rental_days= ('rental_days', 'mean')
).round(1)

,num,avg_rental_days
bike_type,,
Cargo bike,44,3.4
City bike,232,2.9
E-bike,122,3.0


Average rental days does not seem to wary significantly between the different bike types.